In [3]:
import pandas as pd
import requests

In [5]:


url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
}

html = requests.get(url, headers=headers).text

sp500_table = pd.read_html(html)[0]

# Keep only relevant columns
sp500_companies = sp500_table[['Symbol', 'Security', 'GICS Sector']]

sp500_companies.head()





C:\Users\HomePC\AppData\Local\Temp\ipykernel_20048\1237553054.py:9: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  sp500_table = pd.read_html(html)[0]


,Symbol,Security,GICS Sector
0,MMM,3M,Industrials
1,AOS,A. O. Smith,Industrials
2,ABT,Abbott Laboratories,Health Care
3,ABBV,AbbVie,Health Care
4,ACN,Accenture,Information Technology


In [6]:
sp500_companies.to_csv("sp500_companies.csv", index=False)


In [7]:
sector_df = pd.read_csv("sp500_companies.csv")
sector_df.head()


,Symbol,Security,GICS Sector
0,MMM,3M,Industrials
1,AOS,A. O. Smith,Industrials
2,ABT,Abbott Laboratories,Health Care
3,ABBV,AbbVie,Health Care
4,ACN,Accenture,Information Technology


In [8]:
sector_df = sector_df[['Security', 'GICS Sector']]

sector_df.rename(columns={
    'Security': 'Company_CSR',
    'GICS Sector': 'Sector'
}, inplace=True)

sector_df['Company_CSR'] = sector_df['Company_CSR'].str.lower().str.strip()


In [10]:

# Load your final Tableau dataset
tableau_df = pd.read_csv("final_tableau_dataset.csv")

# Load S&P 500 sector data
sp500_df = pd.read_csv("sp500_companies.csv")

print(tableau_df.columns)
print(sp500_df.columns)


Index(['Company_CSR', 'avg_sentiment', 'sentiment_sq', 'e_score', 's_score',
       'g_score', 'total_score', 'post_count'],
      dtype='object')
Index(['Symbol', 'Security', 'GICS Sector'], dtype='object')


In [11]:
tableau_df.head(10)

,Company_CSR,avg_sentiment,sentiment_sq,e_score,s_score,g_score,total_score,post_count
0,boston scientific,0.126595,0.016026,3.16,18.00,11.83,32.98,1.0
1,boston scientific,0.126595,0.016026,2.83,12.86,10.32,26.02,1.0
2,extra space storage,0.108850,0.011848,3.81,4.28,5.86,13.94,1.0
3,archer daniels midland,NaN,NaN,16.38,14.20,5.90,36.36,NaN
4,archer daniels midland,NaN,NaN,15.89,13.51,5.38,34.78,NaN
5,archer daniels midland,NaN,NaN,15.91,13.49,4.85,34.25,NaN
6,best buy,0.176610,0.031191,2.11,5.20,4.60,11.92,17.0
7,apple inc.,0.101427,0.010287,0.55,13.58,10.47,24.32,15.0
8,apple inc.,0.101427,0.010287,0.18,7.69,8.86,16.72,15.0
9,apple inc.,0.101427,0.010287,0.65,6.86,8.95,16.45,15.0


In [12]:
def clean_company_names(series):
    return (
        series
        .str.lower()
        .str.replace("&", "and", regex=False)
        .str.replace(",", "", regex=False)
        .str.replace(".", "", regex=False)
        .str.strip()
    )

tableau_df["Company_clean"] = clean_company_names(tableau_df["Company_CSR"])
sp500_df["Company_clean"] = clean_company_names(sp500_df["Security"])


In [13]:
sector_df = sp500_df[["Company_clean", "GICS Sector"]].copy()
sector_df.rename(columns={"GICS Sector": "Sector"}, inplace=True)


In [14]:
tableau_df = tableau_df.merge(
    sector_df,
    on="Company_clean",
    how="left"
)


In [15]:
print(tableau_df["Sector"].value_counts(dropna=False))


Sector
Financials                135
Health Care               116
Information Technology    109
Industrials               105
Consumer Discretionary     74
Consumer Staples           60
Real Estate                56
NaN                        55
Materials                  50
Utilities                  47
Energy                     44
Communication Services     15
Name: count, dtype: int64
